## SECTION 1 - LOAD DATA

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings("ignore")

In [4]:
app = pd.read_csv("C:/Users/minhn/Downloads/project1-home-credit-risk/dataset/application_train.csv")
bureau = pd.read_csv("C:/Users/minhn/Downloads/project1-home-credit-risk/dataset/bureau.csv")
prev = pd.read_csv("C:/Users/minhn/Downloads/project1-home-credit-risk/dataset/previous_application.csv")
inst = pd.read_csv("C:/Users/minhn/Downloads/project1-home-credit-risk/dataset/installments_payments.csv")

In [5]:
print(app.shape)
print(bureau.shape)
print(prev.shape)
print(inst.shape)

(307511, 122)
(1716428, 17)
(1670214, 37)
(13605401, 8)


In [6]:
app["TARGET"].value_counts(normalize=True)

TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

## SECTION 2 - AGGREGATE FEATURES

### Bureau

In [13]:
bureau["ACTIVE_FLAG"] = (
    bureau["CREDIT_ACTIVE"] == "Active"
).astype(int)

active_loan_cnt = (
    bureau.groupby("SK_ID_CURR")
    ["ACTIVE_FLAG"]
    .sum()
    .reset_index(name="ACTIVE_LOAN_CNT")
)

In [15]:
bureau["OVERDUE_FLAG"] = (
    bureau["CREDIT_DAY_OVERDUE"] > 0
).astype(int)

overdue_loan_cnt = (
    bureau.groupby("SK_ID_CURR")
    ["OVERDUE_FLAG"]
    .sum()
    .reset_index(name="OVERDUE_LOAN_CNT")
)

In [17]:
avg_credit_sum = (
    bureau.groupby("SK_ID_CURR")
    ["AMT_CREDIT_SUM"]
    .mean()
    .reset_index(name="AVG_CREDIT_SUM")
)

### Previous Application

In [20]:
prev_application_cnt = (
    prev.groupby("SK_ID_CURR")
    .size()
    .reset_index(name="PREV_APPLICATION_CNT")
)

In [22]:
prev["APPROVED_FLAG"] = (
    prev["NAME_CONTRACT_STATUS"] == "Approved"
).astype(int)

prev_approved_cnt = (
    prev.groupby("SK_ID_CURR")
    ["APPROVED_FLAG"]
    .sum()
    .reset_index(name="PREV_APPROVED_CNT")
)

In [24]:
approval_rate = (
    prev_application_cnt
    .merge(
        prev_approved_cnt,
        on="SK_ID_CURR",
        how="left"
    )
)

approval_rate["APPROVAL_RATE"] = (
    approval_rate["PREV_APPROVED_CNT"]
    /
    approval_rate["PREV_APPLICATION_CNT"]
)

### Installment

In [27]:
inst["DPD"] = (
    inst["DAYS_ENTRY_PAYMENT"]
    -
    inst["DAYS_INSTALMENT"]
)

In [29]:
avg_dpd = (
    inst.groupby("SK_ID_CURR")
    ["DPD"]
    .mean()
    .reset_index(name="AVG_DPD")
)

In [31]:
max_dpd = (
    inst.groupby("SK_ID_CURR")
    ["DPD"]
    .max()
    .reset_index(name="MAX_DPD")
)

In [33]:
inst["LATE_FLAG"] = (
    inst["DPD"] > 0
).astype(int)

late_payment_cnt = (
    inst.groupby("SK_ID_CURR")
    ["LATE_FLAG"]
    .sum()
    .reset_index(name="LATE_PAYMENT_CNT")
)

## SECTION 3 - MERGE

In [36]:
model_df = app.copy()

In [38]:
model_df = model_df.merge(
    active_loan_cnt,
    on="SK_ID_CURR",
    how="left"
)

model_df = model_df.merge(
    overdue_loan_cnt,
    on="SK_ID_CURR",
    how="left"
)

model_df = model_df.merge(
    avg_credit_sum,
    on="SK_ID_CURR",
    how="left"
)

model_df = model_df.merge(
    approval_rate[
        [
            "SK_ID_CURR",
            "PREV_APPLICATION_CNT",
            "PREV_APPROVED_CNT",
            "APPROVAL_RATE"
        ]
    ],
    on="SK_ID_CURR",
    how="left"
)

model_df = model_df.merge(
    avg_dpd,
    on="SK_ID_CURR",
    how="left"
)

model_df = model_df.merge(
    max_dpd,
    on="SK_ID_CURR",
    how="left"
)

model_df = model_df.merge(
    late_payment_cnt,
    on="SK_ID_CURR",
    how="left"
)

## SECTION 4 - MISSING VALUE TREATMENT

In [46]:
numeric_cols = model_df.select_dtypes(
    include=["int64","float64"]
).columns

categorical_cols = model_df.select_dtypes(
    include=["object"]
).columns

In [48]:
for col in numeric_cols:

    model_df[col] = (
        model_df[col]
        .fillna(
            model_df[col].median()
        )
    )

In [50]:
for col in categorical_cols:

    model_df[col] = (
        model_df[col]
        .fillna(
            model_df[col].mode()[0]
        )
    )

## SECTION 5 - FEATURE ENGINEERING

In [55]:
model_df["EXT_SOURCE_AVG"] = (
    model_df[
        [
            "EXT_SOURCE_1",
            "EXT_SOURCE_2",
            "EXT_SOURCE_3"
        ]
    ]
    .mean(axis=1)
)

In [57]:
model_df["CREDIT_INCOME_RATIO"] = (
    model_df["AMT_CREDIT"]
    /
    model_df["AMT_INCOME_TOTAL"]
)

In [59]:
model_df["ANNUITY_INCOME_RATIO"] = (
    model_df["AMT_ANNUITY"]
    /
    model_df["AMT_INCOME_TOTAL"]
)

## SECTION 6 - MODELING

In [62]:
y = model_df["TARGET"]

X = model_df.drop(
    columns=["TARGET"]
)

In [64]:
X_encoded = pd.get_dummies(
    X,
    drop_first=True
)

In [66]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

### Logistic Regression

In [69]:
lr = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

lr.fit(
    X_train,
    y_train
)

lr_pd = lr.predict_proba(
    X_test
)[:,1]

auc_lr = roc_auc_score(
    y_test,
    lr_pd
)

print(auc_lr)

0.6163769721026777


### XGBoost

In [71]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(
    X_train,
    y_train
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

In [77]:
xgb_pd = xgb.predict_proba(
    X_test
)[:,1]

auc_xgb = roc_auc_score(
    y_test,
    xgb_pd
)

print(auc_xgb)

0.7697971690973322


## SECTION 7 - PD SCORE

In [79]:
model_df["PD_SCORE"] = (
    xgb.predict_proba(
        X_encoded
    )[:,1]
)

## SECTION 8 - RISK SEGMENT

In [82]:
model_df["RISK_SEGMENT"] = pd.qcut(
    model_df["PD_SCORE"],
    q=3,
    labels=[
        "Low Risk",
        "Medium Risk",
        "High Risk"
    ]
)

In [84]:
segment_check = (
    model_df
    .groupby("RISK_SEGMENT")
    ["TARGET"]
    .agg(
        customer_count="count",
        default_rate="mean"
    )
)

segment_check

,customer_count,default_rate
RISK_SEGMENT,,
Low Risk,102504,0.011434
Medium Risk,102503,0.042506
High Risk,102504,0.188246


In [119]:
model_df["RISK_BAND"] = pd.qcut(
    model_df["PD_SCORE"],
    q=5,
    labels=[
        "A",
        "B",
        "C",
        "D",
        "E"
    ]
)

In [127]:
band_check = (
    model_df
    .groupby("RISK_BAND")
    ["TARGET"]
    .mean()
)

In [131]:
band_check 

RISK_BAND
A    0.006992
B    0.020341
C    0.041641
D    0.083087
E    0.251585
Name: TARGET, dtype: float64

In [86]:
importance = pd.DataFrame({
    "feature": X_encoded.columns,
    "importance": xgb.feature_importances_
})

importance = importance.sort_values(
    "importance",
    ascending=False
)

importance.head(20)

,feature,importance
114,EXT_SOURCE_AVG,0.064879
130,NAME_INCOME_TYPE_Pensioner,0.019143
13,FLAG_EMP_PHONE,0.013747
30,EXT_SOURCE_3,0.013567
135,NAME_EDUCATION_TYPE_Higher education,0.012706
80,FLAG_DOCUMENT_3,0.011963
118,CODE_GENDER_M,0.011302
29,EXT_SOURCE_2,0.010628
138,NAME_EDUCATION_TYPE_Secondary / secondary special,0.009685
134,NAME_INCOME_TYPE_Working,0.009639


## SECTION 9 - PORTFOLIO RISK MART

In [133]:
portfolio_risk_mart = model_df[
    [
        "SK_ID_CURR",
        "TARGET",
        "PD_SCORE",
        "RISK_SEGMENT",
        "RISK_BAND",

        "AMT_INCOME_TOTAL",
        "AMT_CREDIT",
        "AMT_ANNUITY",

        "EXT_SOURCE_AVG",

        "CREDIT_INCOME_RATIO",
        "ANNUITY_INCOME_RATIO",

        "ACTIVE_LOAN_CNT",
        "OVERDUE_LOAN_CNT",
        "AVG_CREDIT_SUM",

        "PREV_APPLICATION_CNT",
        "PREV_APPROVED_CNT",
        "APPROVAL_RATE",

        "AVG_DPD",
        "MAX_DPD",
        "LATE_PAYMENT_CNT",

        "CODE_GENDER",
        "NAME_INCOME_TYPE",
        "NAME_EDUCATION_TYPE",
        "NAME_FAMILY_STATUS",
        "OCCUPATION_TYPE"
    ]
]

## SECTION 10 - EXPORT

In [135]:
portfolio_risk_mart.to_csv(
    "portfolio_risk_mart.csv",
    index=False
)

## APPLY TO application_test.csv

In [95]:
app_test = pd.read_csv("C:/Users/minhn/Downloads/project1-home-credit-risk/dataset/application_test.csv")

In [97]:
app_test["EXT_SOURCE_AVG"] = (
    app_test[
        [
            "EXT_SOURCE_1",
            "EXT_SOURCE_2",
            "EXT_SOURCE_3"
        ]
    ].mean(axis=1)
)

app_test["CREDIT_INCOME_RATIO"] = (
    app_test["AMT_CREDIT"]
    /
    app_test["AMT_INCOME_TOTAL"]
)

app_test["ANNUITY_INCOME_RATIO"] = (
    app_test["AMT_ANNUITY"]
    /
    app_test["AMT_INCOME_TOTAL"]
)

In [99]:
test_df = app_test.copy()

test_df = test_df.merge(
    active_loan_cnt,
    on="SK_ID_CURR",
    how="left"
)

test_df = test_df.merge(
    overdue_loan_cnt,
    on="SK_ID_CURR",
    how="left"
)

test_df = test_df.merge(
    avg_credit_sum,
    on="SK_ID_CURR",
    how="left"
)

test_df = test_df.merge(
    approval_rate[
        [
            "SK_ID_CURR",
            "PREV_APPLICATION_CNT",
            "PREV_APPROVED_CNT",
            "APPROVAL_RATE"
        ]
    ],
    on="SK_ID_CURR",
    how="left"
)

test_df = test_df.merge(
    avg_dpd,
    on="SK_ID_CURR",
    how="left"
)

test_df = test_df.merge(
    max_dpd,
    on="SK_ID_CURR",
    how="left"
)

test_df = test_df.merge(
    late_payment_cnt,
    on="SK_ID_CURR",
    how="left"
)

In [101]:
for col in numeric_cols:
    if col in test_df.columns:
        test_df[col] = test_df[col].fillna(
            model_df[col].median()
        )

In [103]:
for col in categorical_cols:
    if col in test_df.columns:
        test_df[col] = test_df[col].fillna(
            model_df[col].mode()[0]
        )

In [105]:
test_features = test_df.drop(
    columns=["SK_ID_CURR"]
)

In [107]:
test_encoded = pd.get_dummies(
    test_features,
    drop_first=True
)

In [109]:
test_encoded = test_encoded.reindex(
    columns=X_encoded.columns,
    fill_value=0
)

In [111]:
test_pd = xgb.predict_proba(
    test_encoded
)[:,1]

In [113]:
submission = pd.DataFrame({
    "SK_ID_CURR": app_test["SK_ID_CURR"],
    "TARGET": test_pd
})

In [115]:
submission

,SK_ID_CURR,TARGET
0,100001,0.030598
1,100005,0.167280
2,100013,0.023041
3,100028,0.045477
4,100038,0.200515
...,...,...
48739,456221,0.029299
48740,456222,0.040151
48741,456223,0.019461
48742,456224,0.058799


In [117]:
submission.to_csv(
    "submission.csv",
    index=False
)

In [137]:
importance.to_csv(
    "feature_importance.csv",
    index=False
)